<div style="font-family:system-ui;border-left:6px solid #B8721A;padding:6px 16px">
<div style="font-size:12px;letter-spacing:.12em;text-transform:uppercase;color:#566273">The AI-Ready Organisation · Lab 6 · Module 6 · ≈30 min</div>
<div style="font-size:26px;font-weight:800;margin:6px 0">Workforce readiness for AI change</div>
<div style="font-size:15px"><b>You will:</b> segment a workforce by role, tenure and satisfaction, build a simple change-risk index, and decide where reskilling and change effort should go first.</div>
<div style="font-size:13px;color:#566273;margin-top:6px">Dataset: IBM HR Analytics Employee Attrition (1,470 employees; fictional data created by IBM data scientists)</div>
</div>

**How to use this notebook** · No coding needed.
1. In the menu choose **Runtime → Run all** (in Kaggle: **Run All**). The first run takes 1–3 minutes.
2. Change values in the **form fields** on the right of each cell (sliders, dropdowns). The cell reruns when you change a value; if it does not, press the play button.
3. Read the plain-English findings under each chart and record your answers in the final cell.

If the real dataset cannot be downloaded (for example, without a Kaggle sign-in), the notebook automatically switches to a **realistic simulated copy with the same structure** and tells you so. The lessons still hold.

In [ ]:
#@title Setup (run first) { display-mode: "form" }
import warnings, os, io, zipfile, urllib.request
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
plt.rcParams.update({"figure.figsize": (9, 4.4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "font.size": 11, "axes.titleweight": "bold"})
BUILD, BUY, BLEND, INK, GOOD, BAD = "#B8721A", "#13807B", "#4255B0", "#111822", "#2D7A4C", "#B03A3A"
pd.set_option("display.max_columns", 30)

def say(text):
    display(Markdown(text))

def scorecard(title, rows):
    """rows: list of (label, value, note)"""
    body = "".join(f"<tr><td style='padding:6px 12px'>{a}</td><td style='padding:6px 12px;font-weight:700;text-align:right'>{b}</td>"
                   f"<td style='padding:6px 12px;color:#566273'>{c}</td></tr>" for a, b, c in rows)
    display(HTML(f"<div style='font-family:system-ui;margin:8px 0'><div style='font-weight:700;font-size:15px;margin-bottom:4px'>{title}</div>"
                 f"<table style='border-collapse:collapse;font-size:14px'>{body}</table></div>"))

def try_kagglehub(handle, filename):
    """Download a public Kaggle dataset. Returns a file path or None."""
    try:
        try:
            import kagglehub
        except ImportError:
            os.system("pip -q install kagglehub")
            import kagglehub
        path = kagglehub.dataset_download(handle)
        for root, _, files in os.walk(path):
            if filename in files:
                return os.path.join(root, filename)
    except Exception as e:
        print(f"Kaggle download unavailable ({type(e).__name__}).")
    return None

RNG = np.random.default_rng(42)
DATA_NOTE = ""

## 1 · Load the workforce data

In [ ]:
#@title Load data { display-mode: "form" }
URL = "https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv"
try:
    df = pd.read_csv(URL, encoding="utf-8-sig"); DATA_NOTE = "IBM HR Analytics data (fictional employees created by IBM)."
except Exception:
    n = 1470
    dept = RNG.choice(["Research & Development", "Sales", "Human Resources"], n, p=[.65, .3, .05])
    roles = {"Research & Development": ["Research Scientist", "Laboratory Technician", "Manufacturing Director", "Healthcare Representative", "Research Director"],
             "Sales": ["Sales Executive", "Sales Representative", "Manager"], "Human Resources": ["Human Resources", "Manager"]}
    role = [RNG.choice(roles[x]) for x in dept]; ot = RNG.choice(["Yes", "No"], n, p=[.28, .72])
    js = RNG.integers(1, 5, n); yrs = RNG.integers(0, 30, n)
    logit = -2.2 + 1.3*(ot == "Yes") - .3*(js-2.5) - .06*yrs + .9*np.isin(role, ["Sales Representative", "Laboratory Technician"])
    df = pd.DataFrame({"Department": dept, "JobRole": role, "OverTime": ot, "JobSatisfaction": js, "EnvironmentSatisfaction": RNG.integers(1, 5, n),
                       "WorkLifeBalance": RNG.integers(1, 5, n), "TrainingTimesLastYear": RNG.integers(0, 7, n), "YearsAtCompany": yrs,
                       "JobLevel": RNG.integers(1, 6, n), "EducationField": RNG.choice(["Life Sciences", "Medical", "Marketing", "Technical Degree", "Other"], n),
                       "Attrition": np.where(RNG.random(n) < 1/(1+np.exp(-logit)), "Yes", "No")})
    DATA_NOTE = "Simulated copy (download failed). Same columns as the IBM dataset."
df["left"] = (df.Attrition == "Yes").astype(int)
df["Tenure band"] = pd.cut(df.YearsAtCompany, [-1, 2, 5, 10, 50], labels=["0–2 yrs", "3–5 yrs", "6–10 yrs", "10+ yrs"]).astype(str)
say(f"**{DATA_NOTE}** {len(df):,} employees; overall attrition **{df.left.mean():.0%}**.")
display(df[["Department", "JobRole", "OverTime", "JobSatisfaction", "TrainingTimesLastYear", "YearsAtCompany", "Attrition"]].head())

## 2 · Explore by group
Choose a department and how to group people. The change-risk index combines four signals that predict resistance or exits during a transformation: attrition, low job satisfaction, overtime load and little recent training.

In [ ]:
#@title Segment the workforce { display-mode: "form" }
department = "All" #@param ["All", "Research & Development", "Sales", "Human Resources"]
group_by = "JobRole" #@param ["JobRole", "Tenure band", "JobLevel", "OverTime", "EducationField"]
d = df if department == "All" else df[df.Department == department]
g = d.groupby(group_by).agg(people=("left", "size"), attrition=("left", "mean"), low_satisfaction=("JobSatisfaction", lambda s: (s <= 2).mean()),
                            overtime=("OverTime", lambda s: (s == "Yes").mean()), low_training=("TrainingTimesLastYear", lambda s: (s <= 1).mean()))
g = g[g.people >= 10]
z = lambda s: (s - s.min()) / (s.max() - s.min() + 1e-9)
g["change_risk_index"] = (100*(.35*z(g.attrition) + .25*z(g.low_satisfaction) + .2*z(g.overtime) + .2*z(g.low_training))).round(0)
g = g.sort_values("change_risk_index", ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(12, max(3.2, .45*len(g)+1)))
axes[0].barh(g.index.astype(str)[::-1], g.change_risk_index[::-1], color=[BAD if i < 2 else BLEND for i in range(len(g))][::-1])
axes[0].set(title="Change-risk index (0–100)", xlabel="Higher = more change effort needed")
heat = g[["attrition", "low_satisfaction", "overtime", "low_training"]]
im = axes[1].imshow(heat.to_numpy(), cmap="Oranges", aspect="auto")
axes[1].set_xticks(range(4), ["Attrition", "Low satisfaction", "Overtime", "Low training"], rotation=20); axes[1].set_yticks(range(len(g)), g.index.astype(str)); axes[1].grid(False)
for i in range(len(g)):
    for j in range(4): axes[1].text(j, i, f"{heat.iat[i, j]:.0%}", ha="center", va="center", fontsize=9)
axes[1].set_title("What drives the index"); plt.tight_layout(); plt.show()
display((g.style.format({"attrition": "{:.0%}", "low_satisfaction": "{:.0%}", "overtime": "{:.0%}", "low_training": "{:.0%}"})))
top = g.index[:2].astype(str).tolist()
say(f"### Highest change risk: **{top[0]}** and **{top[1] if len(top) > 1 else '—'}**\nStart engagement, reskilling and early involvement in AI design with these groups.")

## 3 · Draft a change plan for the top group

In [ ]:
#@title Change plan builder { display-mode: "form" }
ai_impact_on_group = "Tasks augmented by a copilot" #@param ["Tasks augmented by a copilot", "Tasks partly automated", "Role substantially redesigned"]
months_to_launch = 6 #@param {type:"slider", min:2, max:18, step:1}
row = g.iloc[0]
actions = []
if row.low_training > .3: actions.append("Fund hands-on AI training before launch, not after; pair each person with a trained peer.")
if row.overtime > .3: actions.append("Release time for learning: reduce workload during pilot weeks, or adoption will lose to the day job.")
if row.low_satisfaction > .35: actions.append("Involve this group in designing the workflow; co-design builds ownership where satisfaction is low.")
if row.attrition > df.left.mean(): actions.append("Communicate career paths early; uncertainty about roles accelerates exits.")
if ai_impact_on_group == "Role substantially redesigned": actions.append("Agree a redeployment and reskilling commitment with HR and employee representatives.")
actions.append("Track a leading indicator monthly: share of the group actively using the AI tool.")
say(f"**Group:** {g.index[0]} · **AI impact:** {ai_impact_on_group.lower()} · **Launch in:** {months_to_launch} months")
for i, a in enumerate(actions, 1): say(f"{i}. {a}")

## Your reflection → capstone §6
Answer in the form fields, then run the cell. Copy the summary into §6 of your business case on the course page.

In [ ]:
#@title Record your answers { display-mode: "form" }
q1 = "" #@param {type:"string"}
q2 = "" #@param {type:"string"}
q3 = "" #@param {type:"string"}
answers = [("The two groups in my organisation I expect to need most change effort, and why:", q1), ("My operating model choice (centralised, hub-and-spoke, federated) and the cadences I will run:", q2), ("My change plan in three actions:", q3)]
say("### Lab 6 reflection")
for q, a in answers:
    say(f"**{q}**  \n{a if a.strip() else '_(not answered yet)_'}")